# Entregable 1 · Detección de intrusiones en redes de drones con datos tabulares

**Estado del arte, análisis exploratorio e implementación de los tres modelos base de la literatura**

| | |
|---|---|
| **Autor** | Donny Torres |
| **Programa** | Maestría en Analítica de Datos, Universidad del Norte (Barranquilla) |
| **Asignatura** | Deep Learning |
| **Fecha** | Noviembre de 2026 |
| **Conjunto de datos** | *UAVIDS-2025*, versión 1 (`UAVIDS-2025.csv`) |
| **Fuente** | Kaggle: <https://www.kaggle.com/datasets/qinglizeng1997/uavids-2025>; Zenodo, licencia CC BY 4.0, DOI <https://doi.org/10.5281/zenodo.15336998> |
| **Tipo de problema** | Clasificación con datos tabulares, multiclase de cinco categorías |
| **Repositorio** | <https://github.com/DTL-DA/proyecto-uavids-2025> |
| **Jupyter Book** | Pendiente de publicación |

Este cuaderno desarrolla el primer entregable del proyecto final de Deep Learning. El propósito es establecer, con evidencia estadística, el nivel mínimo de desempeño (*baseline*) que cualquier modelo avanzado deberá superar al detectar intrusiones en redes de enjambres de drones, de modo que una mejora futura se pueda juzgar por su relevancia y no solo por un promedio más alto. Primero se define el problema, luego se audita el conjunto de datos, se extraen de la literatura los tres modelos de Deep Learning más básicos y se implementan bajo un mismo protocolo, para cerrar con el baseline y su inferencia estadística.

El proyecto tiene tres entregables encadenados: este fija el baseline con tres modelos, el segundo lo amplía a un benchmark de diez modelos y el tercero lleva el resultado a producción con un modelo original. Por eso la partición, el pipeline y las decisiones de este cuaderno se guardan para reutilizarlos.

## Configuración del entorno

El cuaderno se ejecuta de principio a fin en CPU, porque el conjunto de datos es tabular y pesa unos 20 MB. Se apoya en tres reglas:

- **Los datos se obtienen con un solo paso.** Si el archivo no está en la carpeta del proyecto, se descarga de Kaggle, y en cualquier caso se verifica su suma SHA-256.
- **Cada corrida se guarda al terminar.** Si la ejecución se interrumpe, se vuelve a ejecutar y se salta lo que ya está hecho.
- **Las semillas y la configuración quedan registradas**, junto con las versiones de las librerías, para que cada resultado se pueda reproducir.

In [1]:
import warnings
import hashlib
import json
import os
import random
import subprocess
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.io.formats.style import Styler

# Las tablas del informe usan coma decimal, igual que el texto
pd.set_option("styler.format.decimal", ",")
pd.set_option("styler.format.precision", 4)
pd.set_option("display.float_format", lambda v: f"{v:.5f}".rstrip("0").rstrip(".").replace(".", ","))
_formato_tabla = Styler.format


def formato_con_coma(self, *argumentos, **opciones):
    opciones.setdefault("decimal", ",")
    return _formato_tabla(self, *argumentos, **opciones)


Styler.format = formato_con_coma

RAIZ = Path("Entregable_1_UAVIDS_2025")
CARPETA_DATOS = RAIZ / "datos"
RESULTADOS = RAIZ / "resultados"
CSV = CARPETA_DATOS / "UAVIDS-2025.csv"
SHA_ESPERADO = "d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0"
URL = "https://www.kaggle.com/api/v1/datasets/download/qinglizeng1997/uavids-2025"

CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
RESULTADOS.mkdir(parents=True, exist_ok=True)
if not CSV.exists():
    comprimido = CARPETA_DATOS / "archive.zip"
    subprocess.run(["curl", "-L", "-sS", "-o", str(comprimido), URL], check=True)
    with zipfile.ZipFile(comprimido) as z:
        z.extract("UAVIDS-2025.csv", CARPETA_DATOS)
    comprimido.unlink()
sha = hashlib.sha256(CSV.read_bytes()).hexdigest()
assert sha == SHA_ESPERADO, "El SHA-256 no coincide: la copia está dañada o cambió de versión"
datos = pd.read_csv(CSV)
print("SHA-256:", sha)
print("Forma del archivo:", datos.shape)

SHA-256: d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0
Forma del archivo: (122171, 23)


In [2]:
import platform
from importlib.metadata import version

import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_predict, train_test_split
from statsmodels.stats.multitest import multipletests

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

IDENTIFICADORES = ["FlowID", "SrcAddr", "SrcPort", "DstAddr", "DstPort", "Protocol"]
ATRIBUTOS = ["FlowDuration/s", "TxPackets", "RxPackets", "LostPackets", "TxBytes", "RxBytes", "TxPacketRate/s", "RxPacketRate/s", "TxByteRate/s",
             "RxByteRate/s", "MeanDelay/s", "MeanJitter/s", "Throughput/Kbps", "MeanPacketSize", "PacketDropRate", "AverageHopCount"]
nombre_clase = datos["label"].str.replace(" Attack", "").str.replace(" Traffic", "")
CLASES = sorted(nombre_clase.unique())
K = len(CLASES)
N = len(datos)
y = nombre_clase.map({c: i for i, c in enumerate(CLASES)}).to_numpy()
x_crudo = datos[ATRIBUTOS].to_numpy(dtype=np.float64)


def num(valor, decimales=3):
    '''Da formato al número para el texto: coma decimal y espacio de miles.'''
    return f"{valor:,.{decimales}f}".replace(",", " ").replace(".", ",")


def enumerar(elementos):
    '''Une una lista para el texto: «a, b y c».'''
    elementos = list(elementos)
    return ", ".join(elementos[:-1]) + " y " + elementos[-1] if len(elementos) > 1 else "".join(elementos)


def valor_p(p):
    return "p < 0,001" if p < 0.001 else f"p = {num(p, 3)}"


def interpretar(texto):
    display(Markdown(texto))


def cramer(tabla):
    '''V de Cramér: tamaño de efecto de una tabla de contingencia, entre 0 y 1.'''
    chi2 = stats.chi2_contingency(tabla)[0]
    return float(np.sqrt(chi2 / (tabla.sum() * (min(tabla.shape) - 1))))


# Las gráficas van en escala de grises: las series se distinguen por tono, tipo de línea y marcador
NEGRO, GRIS_OSCURO, GRIS_MEDIO, GRIS_CLARO = "#000000", "#4a4a4a", "#7a7a7a", "#d9d9d9"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110, "image.cmap": "Greys", "axes.grid": True, "grid.color": GRIS_CLARO,
                     "axes.edgecolor": GRIS_MEDIO, "axes.prop_cycle": mpl.cycler(color=[NEGRO, GRIS_OSCURO, GRIS_MEDIO]),
                     "axes.titlesize": 10.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8})

PAQUETES = ["torch", "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib"]
display(pd.DataFrame({"Componente": ["Python", *PAQUETES, "Dispositivo"],
                      "Versión": [platform.python_version(), *map(version, PAQUETES), f"CPU {platform.processor()}, {torch.get_num_threads()} hilos"]}).style.hide(axis="index"))

Componente,Versión
Python,3.13.14
torch,2.6.0+cpu
numpy,2.4.6
pandas,3.0.3
scipy,1.17.1
scikit-learn,1.9.0
statsmodels,0.15.0
matplotlib,3.10.9
Dispositivo,"CPU Intel64 Family 6 Model 154 Stepping 4, GenuineIntel, 10 hilos"
